## Dataset Preparation

In [1]:
import pandas as pd
import numpy as np

data = {
    "Monthly_Usage_GB": [5, 12, np.nan, 20, 50, 8, np.nan, 100],
    "Contract_Type": ["Prepaid", "Postpaid", "Postpaid", "Prepaid", "Postpaid", np.nan, "Prepaid", "Postpaid"],
    "Payment_Method": ["UPI", "Card", "Cash", "UPI", "Card", "Cash", np.nan, "UPI"],
    "Customer_Tenure_Months": [2, 12, 6, 24, np.nan, 4, 1, 60],
    "Monthly_Bill": [299, 499, 399, 599, 999, np.nan, 249, 1999],
    "Churn": ["No", "No", "Yes", "No", "Yes", "No", "No", "Yes"]
}

df = pd.DataFrame(data)
df

,Monthly_Usage_GB,Contract_Type,Payment_Method,Customer_Tenure_Months,Monthly_Bill,Churn
0,5.0,Prepaid,UPI,2.0,299.0,No
1,12.0,Postpaid,Card,12.0,499.0,No
2,NaN,Postpaid,Cash,6.0,399.0,Yes
3,20.0,Prepaid,UPI,24.0,599.0,No
4,50.0,Postpaid,Card,NaN,999.0,Yes
5,8.0,NaN,Cash,4.0,NaN,No
6,NaN,Prepaid,NaN,1.0,249.0,No
7,100.0,Postpaid,UPI,60.0,1999.0,Yes


##Handling Missing Values

In [2]:
#missing values
print(df.isnull().sum())
#mean
df["Monthly_Usage_GB"] = df["Monthly_Usage_GB"].fillna(
    df["Monthly_Usage_GB"].mean()
)
#median
df["Customer_Tenure_Months"] = df["Customer_Tenure_Months"].fillna(
    df["Customer_Tenure_Months"].median()
)
df["Monthly_Bill"] = df["Monthly_Bill"].fillna(
    df["Monthly_Bill"].median()
)
#mode
df["Contract_Type"] = df["Contract_Type"].fillna(
    df["Contract_Type"].mode()[0]
)
df["Payment_Method"] = df["Payment_Method"].fillna(
    df["Payment_Method"].mode()[0]
)
print("Dataset after handling the missing values:")
print(df.isnull().sum())

Monthly_Usage_GB          2
Contract_Type             1
Payment_Method            1
Customer_Tenure_Months    1
Monthly_Bill              1
Churn                     0
dtype: int64
Dataset after handling the missing values:
Monthly_Usage_GB          0
Contract_Type             0
Payment_Method            0
Customer_Tenure_Months    0
Monthly_Bill              0
Churn                     0
dtype: int64


## Label Encoding

In [3]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
df["Contract_Type"] = le.fit_transform(df["Contract_Type"])
df["Churn"] = le.fit_transform(df["Churn"])

print(df)


   Monthly_Usage_GB  Contract_Type Payment_Method  Customer_Tenure_Months  \
0               5.0              1            UPI                     2.0   
1              12.0              0           Card                    12.0   
2              32.5              0           Cash                     6.0   
3              20.0              1            UPI                    24.0   
4              50.0              0           Card                     6.0   
5               8.0              0           Cash                     4.0   
6              32.5              1            UPI                     1.0   
7             100.0              0            UPI                    60.0   

   Monthly_Bill  Churn  
0         299.0      0  
1         499.0      0  
2         399.0      1  
3         599.0      0  
4         999.0      1  
5         499.0      0  
6         249.0      0  
7        1999.0      1  


## One-Hot Encoding

In [4]:
df = pd.get_dummies(df, columns=["Payment_Method"], drop_first=True)

df

,Monthly_Usage_GB,Contract_Type,Customer_Tenure_Months,Monthly_Bill,Churn,Payment_Method_Cash,Payment_Method_UPI
0,5.0,1,2.0,299.0,0,False,True
1,12.0,0,12.0,499.0,0,False,False
2,32.5,0,6.0,399.0,1,True,False
3,20.0,1,24.0,599.0,0,False,True
4,50.0,0,6.0,999.0,1,False,False
5,8.0,0,4.0,499.0,0,True,False
6,32.5,1,1.0,249.0,0,False,True
7,100.0,0,60.0,1999.0,1,False,True


##IQR

In [5]:
def apply_iqr(df, column):
    Q1 = df[column].quantile(0.25)
    Q3 = df[column].quantile(0.75)
    IQR = Q3 - Q1

    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    return df[(df[column] >= lower_bound) & (df[column] <= upper_bound)]

In [6]:
num_cols = [
    "Monthly_Usage_GB",
    "Customer_Tenure_Months",
    "Monthly_Bill"
]

for col in num_cols:
    df = apply_iqr(df, col)
df

,Monthly_Usage_GB,Contract_Type,Customer_Tenure_Months,Monthly_Bill,Churn,Payment_Method_Cash,Payment_Method_UPI
0,5.0,1,2.0,299.0,0,False,True
1,12.0,0,12.0,499.0,0,False,False
2,32.5,0,6.0,399.0,1,True,False
5,8.0,0,4.0,499.0,0,True,False
6,32.5,1,1.0,249.0,0,False,True


##Normalization

In [7]:
X = df.drop("Churn", axis=1)
y = df["Churn"]
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()

num_cols = ["Monthly_Usage_GB", "Customer_Tenure_Months", "Monthly_Bill"]

df[num_cols] = scaler.fit_transform(df[num_cols])

df


,Monthly_Usage_GB,Contract_Type,Customer_Tenure_Months,Monthly_Bill,Churn,Payment_Method_Cash,Payment_Method_UPI
0,0.000000,1,0.090909,0.2,0,False,True
1,0.254545,0,1.000000,1.0,0,False,False
2,1.000000,0,0.454545,0.6,1,True,False
5,0.109091,0,0.272727,1.0,0,True,False
6,1.000000,1,0.000000,0.0,0,False,True


##Lasso Regularization (L1 Regularization)

In [10]:
from sklearn.linear_model import Lasso
import pandas as pd

# Features & target (remove target-like columns from X)
X = df.drop(["Monthly_Bill", "Churn"], axis=1)
y = df["Monthly_Bill"]

# Lasso Regression (L1 Regularization)
lasso = Lasso(alpha=0.1)
lasso.fit(X, y)

# Coefficients
lasso_coef = pd.DataFrame({
    "Feature": X.columns,
    "Lasso_Coefficient": lasso.coef_
})

lasso_coef


,Feature,Lasso_Coefficient
0,Monthly_Usage_GB,-0.000000e+00
1,Contract_Type,-3.500000e-01
2,Customer_Tenure_Months,0.000000e+00
3,Payment_Method_Cash,0.000000e+00
4,Payment_Method_UPI,-2.775558e-16
